# MNIST Digit Recognition — Improved CNN Training (Colab)

This notebook trains a **deeper VGG-style CNN** on the MNIST handwritten digit dataset,
augmented with a synthetic **Junk** class (Class 10) to reject random scribbles.

### Improvements

| Aspect | v1 | v2 (This) |
|---|---|---|
| **Architecture** | 2 conv layers (32->64) | 3 VGG-style blocks (32->64->128) |
| **FC Layer** | 128 units | 256 units + BatchNorm |
| **Augmentation** | None | Rotation, Affine, Perspective, Erasing |
| **Junk Patterns** | 4 types, 6K | 8 types, 18K |
| **Optimizer** | Adam | AdamW + weight decay |
| **LR Schedule** | StepLR | Cosine Annealing |
| **Loss** | CrossEntropy | CrossEntropy + Label Smoothing |
| **Epochs** | 10 | 20 |

## 1. Setup & Imports

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset, ConcatDataset
from torchvision import datasets, transforms

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix
from PIL import Image, ImageDraw, ImageFilter
import os
import random
import time
import math

plt.style.use('dark_background')
sns.set_theme(style='darkgrid')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

## 2. Hyperparameters

In [ ]:
BATCH_SIZE = 128
LEARNING_RATE = 0.001
NUM_EPOCHS = 20
NUM_CLASSES = 11

NUM_JUNK_TRAIN = 18000
NUM_JUNK_TEST = 3000

print(f'Batch Size: {BATCH_SIZE}')
print(f'Learning Rate: {LEARNING_RATE}')
print(f'Epochs: {NUM_EPOCHS}')
print(f'Classes: {NUM_CLASSES} (0-9 digits + junk)')
print(f'Junk samples: {NUM_JUNK_TRAIN} train / {NUM_JUNK_TEST} test')

## 3. Junk Dataset (8 Pattern Types)

In [ ]:
class JunkDataset(Dataset):
    """Generates synthetic 28x28 junk images with 8 diverse pattern types."""

    PATTERN_TYPES = [
        'random_strokes', 'noise', 'grid', 'blob',
        'zigzag', 'crosshatch', 'dots', 'curves'
    ]

    def __init__(self, size, transform=None):
        self.size = size
        self.transform = transform

    def __len__(self):
        return self.size

    def _draw_random_strokes(self, draw):
        for _ in range(random.randint(2, 7)):
            x1, y1 = random.randint(0, 24), random.randint(0, 24)
            x2, y2 = random.randint(x1, 27), random.randint(y1, 27)
            width = random.randint(1, 4)
            color = random.randint(120, 255)
            draw.line([(x1, y1), (x2, y2)], fill=color, width=width)

    def _draw_noise(self, img_array):
        density = random.uniform(0.05, 0.3)
        mask = np.random.random((28, 28)) < density
        img_array[mask] = np.random.randint(100, 256, size=mask.sum())
        return img_array

    def _draw_grid(self, draw):
        spacing = random.randint(3, 8)
        color = random.randint(100, 220)
        width = random.randint(1, 2)
        for x in range(0, 28, spacing):
            draw.line([(x, 0), (x, 27)], fill=color, width=width)
        for y in range(0, 28, spacing):
            draw.line([(0, y), (27, y)], fill=color, width=width)

    def _draw_blob(self, draw):
        for _ in range(random.randint(1, 3)):
            cx, cy = random.randint(5, 22), random.randint(5, 22)
            rx, ry = random.randint(3, 10), random.randint(3, 10)
            color = random.randint(130, 255)
            draw.ellipse([cx - rx, cy - ry, cx + rx, cy + ry], fill=color)

    def _draw_zigzag(self, draw):
        points = []
        x = random.randint(0, 5)
        for i in range(random.randint(4, 8)):
            y = random.randint(0, 27)
            points.append((x, y))
            x = min(27, x + random.randint(2, 6))
        color = random.randint(120, 255)
        if len(points) >= 2:
            draw.line(points, fill=color, width=random.randint(1, 3))

    def _draw_crosshatch(self, draw):
        color = random.randint(100, 240)
        width = random.randint(1, 3)
        for _ in range(random.randint(3, 6)):
            x1, y1 = random.randint(0, 27), random.randint(0, 27)
            angle = random.uniform(0, 2 * math.pi)
            length = random.randint(8, 25)
            x2 = int(x1 + length * math.cos(angle))
            y2 = int(y1 + length * math.sin(angle))
            x2, y2 = max(0, min(27, x2)), max(0, min(27, y2))
            draw.line([(x1, y1), (x2, y2)], fill=color, width=width)

    def _draw_dots(self, draw):
        for _ in range(random.randint(8, 30)):
            x, y = random.randint(0, 27), random.randint(0, 27)
            r = random.randint(0, 2)
            color = random.randint(120, 255)
            draw.ellipse([x - r, y - r, x + r, y + r], fill=color)

    def _draw_curves(self, draw):
        for _ in range(random.randint(2, 5)):
            x1 = random.randint(0, 18)
            y1 = random.randint(0, 18)
            x2 = random.randint(x1 + 4, 28)
            y2 = random.randint(y1 + 4, 28)
            start = random.randint(0, 360)
            end = start + random.randint(60, 300)
            color = random.randint(120, 255)
            draw.arc([x1, y1, x2, y2], start=start, end=end,
                     fill=color, width=random.randint(1, 3))

    def __getitem__(self, idx):
        img = Image.new('L', (28, 28), color=0)
        draw = ImageDraw.Draw(img)

        pattern = random.choice(self.PATTERN_TYPES)

        if pattern == 'random_strokes':
            self._draw_random_strokes(draw)
        elif pattern == 'noise':
            img_array = np.array(img)
            img_array = self._draw_noise(img_array)
            img = Image.fromarray(img_array)
        elif pattern == 'grid':
            self._draw_grid(draw)
        elif pattern == 'blob':
            self._draw_blob(draw)
        elif pattern == 'zigzag':
            self._draw_zigzag(draw)
        elif pattern == 'crosshatch':
            self._draw_crosshatch(draw)
        elif pattern == 'dots':
            self._draw_dots(draw)
        elif pattern == 'curves':
            self._draw_curves(draw)

        if random.random() > 0.4:
            img = img.filter(ImageFilter.GaussianBlur(
                radius=random.uniform(0.3, 1.5)
            ))

        if self.transform:
            img = self.transform(img)

        return img, 10


print('JunkDataset defined (8 pattern types).')

## 4. Load & Combine Datasets

In [ ]:
train_transform = transforms.Compose([
    transforms.RandomRotation(15),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1), scale=(0.85, 1.15)),
    transforms.RandomPerspective(distortion_scale=0.2, p=0.3),
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,)),
    transforms.RandomErasing(p=0.1, scale=(0.02, 0.1)),
])

test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,)),
])

mnist_train = datasets.MNIST(root='./data', train=True, download=True, transform=train_transform)
mnist_test = datasets.MNIST(root='./data', train=False, download=True, transform=test_transform)

junk_train = JunkDataset(NUM_JUNK_TRAIN, transform=train_transform)
junk_test = JunkDataset(NUM_JUNK_TEST, transform=test_transform)

train_dataset = ConcatDataset([mnist_train, junk_train])
test_dataset = ConcatDataset([mnist_test, junk_test])

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f'Training samples: {len(train_dataset)} (MNIST: {len(mnist_train)}, Junk: {len(junk_train)})')
print(f'Test samples:     {len(test_dataset)} (MNIST: {len(mnist_test)}, Junk: {len(junk_test)})')

## 5. Visualize Samples

In [ ]:
fig, axes = plt.subplots(2, 8, figsize=(16, 4))
fig.suptitle('Sample MNIST Digits (with augmentation)', fontsize=16, color='white')
for i, ax in enumerate(axes.flat):
    image, label = mnist_train[i]
    ax.imshow(image.squeeze(), cmap='gray')
    ax.set_title(f'Label: {label}', fontsize=10, color='#00ff88')
    ax.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 8, figsize=(16, 4))
fig.suptitle('Sample Junk Images (Class 10)', fontsize=16, color='white')
for i, ax in enumerate(axes.flat):
    image, label = junk_train[i]
    ax.imshow(image.squeeze(), cmap='gray')
    ax.set_title(f'Label: {label}', fontsize=10, color='#ff6090')
    ax.axis('off')
plt.tight_layout()
plt.show()

## 6. Model Architecture

```
Input (1x28x28)
  -> Conv(1->32) -> BN -> ReLU -> Conv(32->32) -> BN -> ReLU -> Pool -> Drop   [32x14x14]
  -> Conv(32->64) -> BN -> ReLU -> Conv(64->64) -> BN -> ReLU -> Pool -> Drop   [64x7x7]
  -> Conv(64->128) -> BN -> ReLU -> Conv(128->128) -> BN -> ReLU -> Pool -> Drop [128x3x3]
  -> Flatten [1152] -> FC(256) -> BN -> ReLU -> Drop -> FC(11)
```

In [ ]:
class MNISTNet(nn.Module):
    """Deeper VGG-style CNN for MNIST digit classification + junk rejection."""

    def __init__(self, num_classes=NUM_CLASSES):
        super(MNISTNet, self).__init__()

        self.conv1a = nn.Conv2d(1, 32, kernel_size=3, padding=1)
        self.bn1a = nn.BatchNorm2d(32)
        self.conv1b = nn.Conv2d(32, 32, kernel_size=3, padding=1)
        self.bn1b = nn.BatchNorm2d(32)

        self.conv2a = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.bn2a = nn.BatchNorm2d(64)
        self.conv2b = nn.Conv2d(64, 64, kernel_size=3, padding=1)
        self.bn2b = nn.BatchNorm2d(64)

        self.conv3a = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.bn3a = nn.BatchNorm2d(128)
        self.conv3b = nn.Conv2d(128, 128, kernel_size=3, padding=1)
        self.bn3b = nn.BatchNorm2d(128)

        self.pool = nn.MaxPool2d(2, 2)
        self.dropout_conv = nn.Dropout2d(0.25)
        self.dropout_fc = nn.Dropout(0.5)

        self.fc1 = nn.Linear(128 * 3 * 3, 256)
        self.bn_fc = nn.BatchNorm1d(256)
        self.fc2 = nn.Linear(256, num_classes)

    def forward(self, x):
        x = F.relu(self.bn1a(self.conv1a(x)))
        x = F.relu(self.bn1b(self.conv1b(x)))
        x = self.pool(x)
        x = self.dropout_conv(x)

        x = F.relu(self.bn2a(self.conv2a(x)))
        x = F.relu(self.bn2b(self.conv2b(x)))
        x = self.pool(x)
        x = self.dropout_conv(x)

        x = F.relu(self.bn3a(self.conv3a(x)))
        x = F.relu(self.bn3b(self.conv3b(x)))
        x = self.pool(x)
        x = self.dropout_conv(x)

        x = x.view(x.size(0), -1)
        x = F.relu(self.bn_fc(self.fc1(x)))
        x = self.dropout_fc(x)
        x = self.fc2(x)
        return x


model = MNISTNet().to(device)
print(model)
print(f'\nTotal parameters: {sum(p.numel() for p in model.parameters()):,}')

## 7. Training Setup

In [ ]:
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS, eta_min=1e-6)

print(f'Loss: CrossEntropyLoss (label_smoothing=0.1)')
print(f'Optimizer: AdamW (lr={LEARNING_RATE}, weight_decay=1e-4)')
print(f'Scheduler: CosineAnnealingLR (T_max={NUM_EPOCHS})')

## 8. Training Loop

In [ ]:
train_losses = []
train_accuracies = []
test_losses = []
test_accuracies = []
best_acc = 0.0
MODEL_PATH = 'mnist_cnn.pth'

for epoch in range(NUM_EPOCHS):
    start = time.time()

    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

    train_loss = running_loss / len(train_loader)
    train_acc = 100. * correct / total
    train_losses.append(train_loss)
    train_accuracies.append(train_acc)

    model.eval()
    test_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            test_loss += loss.item()
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()

    test_loss /= len(test_loader)
    test_acc = 100. * correct / total
    test_losses.append(test_loss)
    test_accuracies.append(test_acc)

    scheduler.step()
    lr = optimizer.param_groups[0]['lr']

    marker = ''
    if test_acc > best_acc:
        best_acc = test_acc
        torch.save(model.state_dict(), MODEL_PATH)
        marker = '  << best'

    elapsed = time.time() - start
    print(f'Epoch [{epoch+1}/{NUM_EPOCHS}] ({elapsed:.1f}s)  '
          f'Train Loss: {train_loss:.4f}  Acc: {train_acc:.2f}%  |  '
          f'Test Loss: {test_loss:.4f}  Acc: {test_acc:.2f}%  |  '
          f'LR: {lr:.6f}{marker}')

print(f'\nTraining complete! Best test accuracy: {best_acc:.2f}%')

## 9. Training Curves

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
epochs_range = range(1, NUM_EPOCHS + 1)

ax1.plot(epochs_range, train_losses, 'o-', color='#00ff88', label='Train Loss', linewidth=2)
ax1.plot(epochs_range, test_losses, 's-', color='#ff6090', label='Test Loss', linewidth=2)
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss')
ax1.set_title('Loss Curve', fontsize=14, color='white')
ax1.legend(fontsize=11); ax1.grid(True, alpha=0.3)

ax2.plot(epochs_range, train_accuracies, 'o-', color='#00e5ff', label='Train Acc', linewidth=2)
ax2.plot(epochs_range, test_accuracies, 's-', color='#ffd740', label='Test Acc', linewidth=2)
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Accuracy (%)')
ax2.set_title('Accuracy Curve', fontsize=14, color='white')
ax2.legend(fontsize=11); ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 10. Confusion Matrix

In [ ]:
model.load_state_dict(torch.load(MODEL_PATH, map_location=device, weights_only=True))
model.eval()

all_preds = []
all_labels = []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        outputs = model(images)
        _, predicted = outputs.max(1)
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.numpy())

cm = confusion_matrix(all_labels, all_preds)
class_labels = [str(i) for i in range(10)] + ['Junk']

fig, ax = plt.subplots(figsize=(11, 9))
sns.heatmap(cm, annot=True, fmt='d', cmap='YlGnBu', ax=ax,
            xticklabels=class_labels, yticklabels=class_labels,
            linewidths=0.5, linecolor='gray')
ax.set_xlabel('Predicted Label', fontsize=13)
ax.set_ylabel('True Label', fontsize=13)
ax.set_title('Confusion Matrix (11 classes)', fontsize=15)
plt.tight_layout()
plt.show()

print('\nPer-class Accuracy:')
print('-' * 30)
for i in range(NUM_CLASSES):
    class_acc = cm[i, i] / cm[i].sum() * 100
    label = f'Digit {i}' if i < 10 else 'Junk   '
    print(f'  {label}: {class_acc:.1f}%')

## 11. Sample Predictions

In [ ]:
model.eval()
fig, axes = plt.subplots(2, 5, figsize=(15, 6))
indices = np.random.choice(len(test_dataset), 10, replace=False)

for i, ax in enumerate(axes.flat):
    image, true_label = test_dataset[indices[i]]
    with torch.no_grad():
        output = model(image.unsqueeze(0).to(device))
        probs = F.softmax(output, dim=1).cpu().numpy()[0]
        pred_label = probs.argmax()
        confidence = probs[pred_label] * 100

    ax.imshow(image.squeeze(), cmap='gray')
    color = '#00ff88' if pred_label == true_label else '#ff4444'
    pred_str = str(pred_label) if pred_label < 10 else 'Junk'
    true_str = str(true_label) if true_label < 10 else 'Junk'
    ax.set_title(f'Pred: {pred_str} ({confidence:.1f}%)\nTrue: {true_str}',
                 fontsize=10, color=color)
    ax.axis('off')

plt.suptitle('Sample Predictions (Green=Correct, Red=Wrong)', fontsize=14, color='white')
plt.tight_layout()
plt.show()

## 12. Download Model (.pth)

Downloads the trained model file to your local machine.
Place it in `model/mnist_cnn.pth` in your project folder.

In [ ]:
from google.colab import files

print(f'Model file: {MODEL_PATH}')
print(f'File size: {os.path.getsize(MODEL_PATH) / 1024:.1f} KB')
print(f'Best test accuracy: {best_acc:.2f}%')
print()
print('Downloading mnist_cnn.pth ...')
print('After download, place it in: model/mnist_cnn.pth')

files.download(MODEL_PATH)